# NB15 — NETL Produced Waters Replication

Tests the primary per-Mb specialization finding in **NETL produced waters** —
hydraulically fractured well samples with extremely high metal concentrations.

**Dataset**: `netl_pw_dna` Spark namespace — 16S V4 amplicon (primers 515F/806R)
from produced water and associated reference environments.

---

## Results

| Block | Niche axis | n_genera | β (total) | p | ABR β | ABR p | λ | Verdict |
|---|---|---|---|---|---|---|---|---|
| 4 (original) | Well type Shannon H (w/ typo) | 363 | −0.007 | 0.527 | — | — | <0 | ❌ Flawed metric |
| 4b (corrected) | Well type Levins' B | 258 | +0.023 | 0.269 | +0.039 | 0.027 | 0.28–0.30 | ❌ Null; ABR also significant |
| 6b (Ba strata) | Barium conc. Levins' B | 73 | +0.045 | 0.004 | — | — | −0.75 | ❌ λ invalid; R script crashed |

## Why NETL is uninformative

**1. Sign reversal with ABR negative control failure (Block 4b)**
All β are positive — more metal genes/Mb predicts broader well-type distribution. The sign is
opposite to the primary result (β=−0.022). Critically, the ABR negative control is also significant
(p=0.027, β=+0.039), with similar magnitude to Tier1 (p=0.035, β=+0.048). This means the signal
is not metal-specific: it reflects a generic "gene-rich genera are broader across hydrocarbon well types"
effect. Genera with more annotated clusters of any type tend to be more cosmopolitan — this confound
overwhelms any metal-specific signal when only 3 niche categories are available.

**2. Niche axis conceptual mismatch**
Well types (Shale Gas, Coal Bed Methane, Tight Oil) classify hydrocarbon source formation, not metal
exposure gradient. The Ba-concentration diagnostic reveals a >1000-fold Ba range across samples
(5–7000 mg/L), yet the well type categories do not capture this gradient: well-type membership is
determined by geology, not geochemistry. The correct niche axis (Block 6) requires per-sample metal
concentration data.

**3. Ba-strata underpowered with model misspecification (Block 6b)**
Only 73 genera have both Ba measurements and GTDB metal data. Lambda=−0.75 for the total model
(outside the valid [0,1] range for Pagel's λ), and the R script crashed on Tier1 due to a degenerate
coefficient table. With only 3 Ba strata and sparse sample overlap, the PGLS is unreliable.

**4. Taxonomy bridge loss**
RDP genus names → GTDB lowercased match loses ~85% of NETL genera: 1,728 RDP genera → 267 with
GTDB metal data (Block 4b) or 77 with Ba data. Names like "Blvii28 wastewater-sludge group" have
no GTDB equivalent. The matched subset is biased toward conserved, cosmopolitan lineages.

## Does this contradict the primary finding?

No. NETL's null is structural. The RDP taxonomy bridge, 3-category niche metric, and absence of
geochemical gradient in the well-type classification make this dataset incapable of detecting the
primary signal with current data. AusMicrobiome (NB13) is a better-structured null: same methodology,
sufficient n, but vegetation-based ecological zones rather than metal-geochemical gradients.

**Status**: ❌ Uninformative — structural limitations prevent valid replication test

## Block 0 — Spark setup

In [1]:
# Hardcoded Spark environment – requires internal cluster access
try:
    spark
except NameError:
    sys.path.append('/opt/conda/lib/python3.13/site-packages')
    from berdl_notebook_utils.setup_spark_session import get_spark_session
    spark = get_spark_session()
print("Spark session ready.")
SPARK = True

Spark session ready.


In [2]:
if SPARK:
    tables = spark.sql('SHOW TABLES IN netl_pw_dna').toPandas()
    print('Tables in netl_pw_dna:')
    print(tables.to_string())
else:
    print('Skipping — no Spark')

Tables in netl_pw_dna:
      namespace                  tableName  isTemporary
0   netl_pw_dna               dna_metadata        False
1   netl_pw_dna          assembly_taxonomy        False
2   netl_pw_dna           amplicon_27f_519        False
3   netl_pw_dna          amplicon_751_1204        False
4   netl_pw_dna          amplicon_926_1392        False
5   netl_pw_dna         amplicon_515f_806r        False
6   netl_pw_dna            study_citations        False
7   netl_pw_dna            study_abundance        False
8   netl_pw_dna               qiime_counts        False
9   netl_pw_dna          sample_attributes        False
10  netl_pw_dna        feature_annotations        False
11  netl_pw_dna      study_sample_metadata        False
12  netl_pw_dna  study_feature_annotations        False


## Block 1 — Schema exploration

In [3]:
if SPARK:
    for tbl in ['amplicon_515f_806r', 'amplicon_27f_519', 'qiime_counts',
                'study_sample_metadata', 'feature_annotations']:
        try:
            print(f'\n=== {tbl} ===')
            desc = spark.sql(f'DESCRIBE netl_pw_dna.{tbl}').toPandas()
            print(desc.to_string())
            cnt = spark.sql(f'SELECT COUNT(*) as n FROM netl_pw_dna.{tbl}').collect()[0].n
            print(f'Rows: {cnt:,}')
        except Exception as e:
            print(f'  ERROR: {e}')


=== amplicon_515f_806r ===


                            col_name data_type comment
0                                 id    string     NaN
1                                HP1    string     NaN
2                                HP2    string     NaN
3                                FWT    string     NaN
4               10Separator_December    string     NaN
5                10Separator_January    string     NaN
6                  10Separator_March    string     NaN
7                10Separator_October    string     NaN
8                    10Tank_December    string     NaN
9                       10Tank_March    string     NaN
10                    10Tank_October    string     NaN
11              11Separator_December    string     NaN
12               11Separator_January    string     NaN
13                 11Separator_March    string     NaN
14               11Separator_October    string     NaN
15                    11Tank_October    string     NaN
16               12Separator_October    string     NaN
17        

Rows: 27,818

=== amplicon_27f_519 ===


           col_name data_type comment
0                id    string     NaN
1      FG-11_coal_1    string     NaN
2      FG-11_coal_2    string     NaN
3      FG-11_coal_3    string     NaN
4       FG-11_lower    string     NaN
5       FG-11_upper    string     NaN
6         Sandstone    string     NaN
7   Sandstone_upper    string     NaN
8       Siltstone_1    string     NaN
9       Siltstone_2    string     NaN
10  Siltstone_lower    string     NaN
11      Arch_HBA-01    string     NaN
12      Arch_HBA-03    string     NaN
13      Arch_HBA-05    string     NaN
14      Arch_HBA-07    string     NaN
15      Arch_HBA-09    string     NaN
16      Arch_HBA-10    string     NaN
17      Arch_HBA-11    string     NaN
18      Arch_HBA-17    string     NaN
19      Arch_HBA-20    string     NaN
20      Arch_HBA-23    string     NaN
21      Arch_HBA-25    string     NaN
22      Arch_HBA-29    string     NaN
23      Arch_HBB-03    string     NaN
24      Arch_HBB-07    string     NaN
25      Arch

Rows: 860

=== qiime_counts ===


...

Rows: 116,365

=== study_sample_metadata ===


    col_name data_type comment
0  sample_id    string     NaN
1      study    string     NaN
2  attribute    string     NaN
3     source    string     NaN
4       unit    string     NaN
5      value    string     NaN


Rows: 25,368

=== feature_annotations ===


              col_name data_type comment
0           feature_id    string     NaN
1          primer_pair    string     NaN
2         rdp_taxonomy    string     NaN
3   faprotax_functions    string     NaN
4  metacyc_predictions    string     NaN
5       source_dataset    string     NaN


Rows: 43,507


In [4]:
if SPARK:
    for tbl in ['amplicon_515f_806r', 'study_sample_metadata', 'feature_annotations']:
        try:
            print(f'\n=== {tbl} (first 3 rows) ===')
            df = spark.sql(f'SELECT * FROM netl_pw_dna.{tbl} LIMIT 3').toPandas()
            print(df.to_string())
        except Exception as e:
            print(f'  ERROR: {e}')


=== amplicon_515f_806r (first 3 rows) ===


                                 id HP1 HP2 FWT 10Separator_December 10Separator_January 10Separator_March 10Separator_October 10Tank_December 10Tank_March 10Tank_October 11Separator_December 11Separator_January 11Separator_March 11Separator_October 11Tank_October 12Separator_October 12Tank_December 12Tank_March 12Tank_October 13Separator_December 13Tank_January 14Separator_December 14Tank_January 15Separator_December 16Tank_December 16Tank_October 17Separator_March 1Separator_December 1Separator_March 1Separator_October 1Tank_December 2Separator_December 2Separator_March 2Tank_December 2Tank_March 3Separator_March 3Separator_October 3Tank_January 4Separator_March 4Tank_December 4Tank_January 4Tank_March 4Tank_October 5Separator_December 5Separator_January 5Separator_March 5Tank_December 5Tank_January 5Tank_March 6Separator_January 6Tank_October 7Separator_January 7Separator_March 7Tank_March 8Separator_March 8Separator_October 8Tank_December 8Tank_March 9Tank_December DJB-1_day_1477 D

  sample_id    study     attribute         source unit                                 value
0  2_TP5_33  An-2013  molec_method  SampleService                                   Amplicon
1  2_TP5_33  An-2013      coordapx  SampleService                             State centroid
2  2_TP5_34  An-2013     sample_id  SampleService       efa4e255-b24f-45f2-8747-d58337fec724

=== feature_annotations (first 3 rows) ===


                         feature_id primer_pair                                                                                   rdp_taxonomy faprotax_functions metacyc_predictions source_dataset
0  9f234317ce5d2498fe1308fc5d30cbec  926F-1392R       Bacteria;Fusobacteriota;Fusobacteriia;Fusobacteriales;Fusobacteriaceae;Psychrilyobacter;                                                      
1  9f275cdc985a1ff3271633a36b5292ff  926F-1392R       Bacteria;Proteobacteria;Alphaproteobacteria;Caulobacterales;Caulobacteraceae;uncultured;                                                      
2  9f2e18a1b5fa6a63bb4779c6dd0a7b1d  926F-1392R  Bacteria;Proteobacteria;Gammaproteobacteria;Oceanospirillales;Nitrincolaceae;Marinobacterium;                                                      


## Block 2 — Taxonomy and genus mapping

In [5]:
if SPARK:
    # Check taxonomy format in feature_annotations
    print('=== feature_annotations taxonomy sample ===')
    try:
        tax_df = spark.sql("""
            SELECT *
            FROM netl_pw_dna.feature_annotations
            LIMIT 20
        """).toPandas()
        print(tax_df.to_string())
        print()
        # Check for genus-level taxonomy columns
        print('Columns:', tax_df.columns.tolist())
    except Exception as e:
        print(f'ERROR: {e}')

=== feature_annotations taxonomy sample ===


                          feature_id primer_pair                                                                                              rdp_taxonomy faprotax_functions                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                           

In [6]:
if SPARK:
    # Try to extract genus from taxonomy string (SILVA format: d__Bacteria;p__...;g__Genus;...)
    try:
        tax_cols = spark.sql('DESCRIBE netl_pw_dna.feature_annotations').toPandas()
        print('feature_annotations columns:', tax_cols['col_name'].tolist())
    except Exception as e:
        print(f'ERROR: {e}')
    print()
    print('Decision gate: if taxonomy is SILVA-based (not GTDB), check genus name overlap')
    print('with primary analysis before proceeding to PGLS.')

feature_annotations columns: ['feature_id', 'primer_pair', 'rdp_taxonomy', 'faprotax_functions', 'metacyc_predictions', 'source_dataset']

Decision gate: if taxonomy is SILVA-based (not GTDB), check genus name overlap
with primary analysis before proceeding to PGLS.


## Block 3 — Sample metadata and environment categories

In [7]:
if SPARK:
    try:
        meta = spark.sql("""
            SELECT *
            FROM netl_pw_dna.study_sample_metadata
            LIMIT 20
        """).toPandas()
        print('study_sample_metadata (20 rows):')
        print(meta.to_string())
        print()
        print('Unique values in potentially useful columns:')
        for col in meta.columns:
            n_unique = meta[col].nunique()
            if 2 <= n_unique <= 30:
                print(f'  {col}: {sorted(meta[col].dropna().unique().tolist())}')
    except Exception as e:
        print(f'ERROR: {e}')

study_sample_metadata (20 rows):
   sample_id    study                attribute         source     unit                                 value
0   2_TP5_33  An-2013             molec_method  SampleService                                       Amplicon
1   2_TP5_33  An-2013                 coordapx  SampleService                                 State centroid
2   2_TP5_34  An-2013                sample_id  SampleService           efa4e255-b24f-45f2-8747-d58337fec724
3   2_TP5_34  An-2013                  version  SampleService                                              1
4   2_TP5_34  An-2013                     type  SampleService                                   BioReplicate
5   2_TP5_34  An-2013                 operator  SampleService                                         Suncor
6   2_TP5_34  An-2013  sesar:depth_in_core_max  SampleService   meters                    20.013123999999998
7   2_TP5_34  An-2013  sesar:depth_in_core_min  SampleService   meters                    20.01

In [8]:
if SPARK:
    # Get all unique environment/sample types for niche metric design
    try:
        env_counts = spark.sql("""
            SELECT *
            FROM netl_pw_dna.study_sample_metadata
        """).toPandas()
        print(f'Total samples: {len(env_counts):,}')
        # Print distribution of any categorical column
        for col in env_counts.columns:
            n_unique = env_counts[col].nunique()
            if 2 <= n_unique <= 20 and env_counts[col].dtype == object:
                print(f'\n{col} ({n_unique} categories):')
                print(env_counts[col].value_counts())
    except Exception as e:
        print(f'ERROR: {e}')

Total samples: 25,368


## Block 4 — Amplicon data extraction

*(Complete after Block 1-3 inspection reveals table structure)*

In [9]:
# Check primer pairs and sample types
if SPARK:
    # Primer pairs in qiime_counts
    print("=== Primer pairs ===")
    spark.sql("SELECT primer_pair, COUNT(*) as n FROM netl_pw_dna.qiime_counts GROUP BY primer_pair").show()
    
    # Check what attributes are available in study_sample_metadata
    print("\n=== Available attributes ===")
    spark.sql("SELECT DISTINCT attribute FROM netl_pw_dna.study_sample_metadata").show(50, truncate=False)

=== Primer pairs ===


+-----------+-----+
|primer_pair|    n|
+-----------+-----+
|  515F-806R|95764|
|   27F-519R| 1725|
| 926F-1392R|11966|
| 751F-1204R| 6910|
+-----------+-----+


=== Available attributes ===


+-----------------------+
|attribute              |
+-----------------------+
|forward_primer         |
|datecomp               |
|doi                    |
|longitude              |
|day                    |
|coordapx               |
|idnewt                 |
|field                  |
|cl                     |
|molec_method           |
|pb                     |
|idorig                 |
|ba                     |
|id-pwdna               |
|acetate                |
|sample_template        |
|h2                     |
|firstauthorlastname    |
|api                    |
|ni                     |
|datesample             |
|idusgs                 |
|era                    |
|k                      |
|nh4                    |
|first_author_last_name |
|mg                     |
|reverse_primer         |
|sr                     |
|latitude               |
|basin                  |
|co                     |
|br                     |
|fetot                  |
|16s_variable_region    |
|sequence_lo

In [10]:
# Check welltype and other potential environment categories
if SPARK:
    # Pivot the EAV metadata to see what categories are available per sample
    print("=== Sample of welltype values ===")
    spark.sql("""
        SELECT sample_id, value as welltype
        FROM netl_pw_dna.study_sample_metadata
        WHERE attribute = 'welltype'
        LIMIT 20
    """).show()
    
    # Also check enigma:method
    print("\n=== enigma:method values ===")
    spark.sql("""
        SELECT sample_id, value as method
        FROM netl_pw_dna.study_sample_metadata
        WHERE attribute = 'enigma:method'
        LIMIT 20
    """).show()
    
    # Count unique values per attribute
    print("\n=== Attributes with value counts ===")
    spark.sql("""
        SELECT attribute, COUNT(DISTINCT value) as n_unique_values, COUNT(*) as n_rows
        FROM netl_pw_dna.study_sample_metadata
        GROUP BY attribute
        ORDER BY n_unique_values DESC
        LIMIT 20
    """).show(20, truncate=False)

=== Sample of welltype values ===


+----------+---------+
| sample_id| welltype|
+----------+---------+
|  2_TP5_34|Tight Oil|
|  2_TP5_35|Tight Oil|
|  2_TP5_36|Tight Oil|
|  2_TP5_37|Tight Oil|
|  2_TP5_38|Tight Oil|
|  2_TP5_39|Tight Oil|
|  2_TP5_40|Tight Oil|
|  2_TP5_41|Tight Oil|
|  2_TP5_42|Tight Oil|
|  2_TP5_43|Tight Oil|
|  3_TP6_44|Tight Oil|
|  3_TP6_45|Tight Oil|
|  3_TP6_46|Tight Oil|
|  3_TP6_47|Tight Oil|
|  3_TP6_48|Tight Oil|
|  3_TP6_49|Tight Oil|
|TF-F-1_Bac|Shale Gas|
|TF-F-2_Bac|Shale Gas|
|    Arc_W1|Shale Gas|
|    Arc_W2|Shale Gas|
+----------+---------+


=== enigma:method values ===


+----------+--------------+
| sample_id|        method|
+----------+--------------+
|  2_TP5_34|Treatment Pond|
|  2_TP5_35|Treatment Pond|
|  2_TP5_36|Treatment Pond|
|  2_TP5_37|Treatment Pond|
|  2_TP5_38|Treatment Pond|
|  2_TP5_39|Treatment Pond|
|  2_TP5_40|Treatment Pond|
|  2_TP5_41|Treatment Pond|
|  2_TP5_42|Treatment Pond|
|  2_TP5_43|Treatment Pond|
|  3_TP6_44|Treatment Pond|
|  3_TP6_45|Treatment Pond|
|  3_TP6_46|Treatment Pond|
|  3_TP6_47|Treatment Pond|
|  3_TP6_48|Treatment Pond|
|  3_TP6_49|Treatment Pond|
|TF-F-1_Bac|  Pumped Water|
|TF-F-2_Bac|  Pumped Water|
|    Arc_W1|      Wellhead|
|    Arc_W2|      Wellhead|
+----------+--------------+


=== Attributes with value counts ===


+-----------------------+---------------+------+
|attribute              |n_unique_values|n_rows|
+-----------------------+---------------+------+
|sample_id              |929            |929   |
|idorig                 |925            |929   |
|id-pwdna               |919            |929   |
|sequence_location      |494            |606   |
|wellname               |246            |663   |
|latitude               |227            |916   |
|longitude              |226            |916   |
|sesar:depth_in_core_max|118            |507   |
|sesar:depth_in_core_min|118            |507   |
|datesample             |91             |716   |
|ca                     |74             |81    |
|na                     |74             |81    |
|ba                     |58             |65    |
|cl                     |47             |48    |
|k                      |34             |40    |
|mg                     |34             |40    |
|idusgs                 |28             |57    |
|doi                

In [11]:
# Block 4 — Build genus-level count matrix from 515F-806R data
if SPARK:
    # The 515F-806R table has feature IDs (from feature_annotations) as rows, 
    # sample IDs as columns. We need to pivot this.
    # First, let's check if qiime_counts is easier to work with
    print("=== qiime_counts structure ===")
    spark.sql("""
        SELECT primer_pair, COUNT(*) as n, 
               COUNT(DISTINCT feature_id) as n_features,
               COUNT(DISTINCT sample_id) as n_samples
        FROM netl_pw_dna.qiime_counts
        GROUP BY primer_pair
    """).show()

=== qiime_counts structure ===


+-----------+-----+----------+---------+
|primer_pair|    n|n_features|n_samples|
+-----------+-----+----------+---------+
|  515F-806R|95764|     27818|      502|
|   27F-519R| 1725|       860|       43|
| 926F-1392R|11966|     11538|      210|
| 751F-1204R| 6910|      3291|       79|
+-----------+-----+----------+---------+



In [12]:
# Use qiime_counts for 515F-806R — it's already in long format
if SPARK:
    # Extract genus-level counts using the taxonomy annotations
    # Join qiime_counts → feature_annotations → genus extraction
    print("Extracting genus-level counts for 515F-806R samples...")
    
    genus_counts = spark.sql("""
        WITH genus_features AS (
            SELECT 
                feature_id,
                -- Extract genus from RDP taxonomy (last semicolon-delimited field before the final semicolon)
                -- Format: Bacteria;Proteobacteria;Gammaproteobacteria;Oceanospirillales;Nitrincolaceae;Marinobacterium;
                TRIM(SPLIT_PART(rdp_taxonomy, ';', 
                     SIZE(SPLIT(rdp_taxonomy, ';')) - 1)) as genus
            FROM netl_pw_dna.feature_annotations
            WHERE primer_pair = '515F-806R'
              AND rdp_taxonomy IS NOT NULL
              AND rdp_taxonomy != ''
        )
        SELECT 
            gf.genus,
            qc.sample_id,
            SUM(qc.count) as total_count
        FROM netl_pw_dna.qiime_counts qc
        JOIN genus_features gf ON qc.feature_id = gf.feature_id
        WHERE qc.primer_pair = '515F-806R'
          AND gf.genus != ''
          AND gf.genus IS NOT NULL
        GROUP BY gf.genus, qc.sample_id
    """)
    
    genus_pd = genus_counts.toPandas()
    print(f"Genus-level counts: {len(genus_pd):,} rows")
    print(f"Unique genera: {genus_pd['genus'].nunique():,}")
    print(f"Unique samples: {genus_pd['sample_id'].nunique():,}")
    print(genus_pd.head(10).to_string())

Extracting genus-level counts for 515F-806R samples...


Genus-level counts: 37,453 rows
Unique genera: 1,728
Unique samples: 495
                             genus                       sample_id  total_count
0  Blvii28 wastewater-sludge group              Low_SH_396_2015_BT          889
1                    Methylotenera           Bac_K-09_BT_Sept_2015            3
2                      Moheibacter                    Bowland-1_68           37
3                   Hyphomicrobium          Bac_N-11_BT_April_2016          517
4                      Bacteroidia          Bac_T-09_H2O_July_2014           38
5                      Aminobacter  Bac_SS-13_BT_April_2016_slurry          632
6                 Rubellimicrobium                  600ml_carboy_2           59
7                            SWB02           Bac_T-09_BT_Sept_2015           11
8                     Syntrophales                 Low_HWC_2013_BT           57
9                         Bacteria                         NV73_T1           34


In [13]:
# Get sample metadata with welltype for environment classification
if SPARK:
    sample_meta = spark.sql("""
        SELECT sample_id, value as welltype
        FROM netl_pw_dna.study_sample_metadata
        WHERE attribute = 'welltype'
          AND value IS NOT NULL
          AND value != ''
    """).toPandas()
    
    # Also get enigma:method for alternative classification
    sample_method = spark.sql("""
        SELECT sample_id, value as method
        FROM netl_pw_dna.study_sample_metadata
        WHERE attribute = 'enigma:method'
          AND value IS NOT NULL
          AND value != ''
    """).toPandas()
    
    print(f"Welltype metadata: {len(sample_meta)} samples")
    print(f"Method metadata: {len(sample_method)} samples")
    print(f"\nWelltype distribution:")
    print(sample_meta['welltype'].value_counts())
    print(f"\nMethod distribution:")
    print(sample_method['method'].value_counts())

Welltype metadata: 916 samples
Method metadata: 899 samples

Welltype distribution:
welltype
Shale Gas           361
Coal Bed Methane    332
Tight Oil           155
Shale_Gas            38
Undefined            30
Name: count, dtype: int64

Method distribution:
method
Separator              280
Pumped Water           190
Downwell Incubation    189
Core                    96
Treatment Pond          56
Wellhead                44
Storage Tank            35
Treater                  5
Treatment Facility       4
Name: count, dtype: int64


## Block 4a — Diagnostic: does well type differ in metal concentration?

Before using well type as the niche axis, validate that the three categories (Shale Gas, Coal Bed Methane, Tight Oil) actually differ in metal concentrations. Only 14% of samples (65–81) have metal measurements, but the distribution across well types is informative.

In [15]:
# Diagnostic: does well type differ in metal concentrations?
# Validates whether welltype can serve as a proxy for metal exposure gradient.
# If medians differ markedly => well type is a defensible proxy.
# If not => metal-concentration niche path (Block 6) is required.
if SPARK:
    metals_long = spark.sql("""
        SELECT sample_id, attribute as metal, CAST(value AS DOUBLE) as conc
        FROM netl_pw_dna.study_sample_metadata
        WHERE attribute IN ('ba','fetot','mn','ni','co','pb')
          AND value IS NOT NULL
          AND TRY_CAST(value AS DOUBLE) IS NOT NULL
    """).toPandas()
    print(f"Metal measurements: {len(metals_long)} rows across {metals_long['sample_id'].nunique()} samples")
    print(metals_long.groupby('metal')['sample_id'].nunique().rename('n_samples').to_frame().T)

    metals_wide = metals_long.pivot_table(
        index='sample_id', columns='metal', values='conc', aggfunc='mean'
    ).reset_index()

    # Fix welltype labels for merge
    sample_meta_diag = sample_meta.copy()
    sample_meta_diag['welltype'] = sample_meta_diag['welltype'].str.strip().replace('Shale_Gas', 'Shale Gas')

    metals_typed = metals_wide.merge(sample_meta_diag, on='sample_id', how='inner')
    print(f"\nSamples with welltype + metal data: {len(metals_typed)}")
    print("Welltype distribution in this subset:")
    print(metals_typed['welltype'].value_counts())
    print()

    for metal in ['ba','fetot','mn','ni']:
        if metal in metals_typed.columns:
            sub = metals_typed[['welltype', metal]].dropna()
            if len(sub) >= 5:
                print(f"\n{metal.upper()} by welltype (n={len(sub)}):")
                print(sub.groupby('welltype')[metal].agg(['count','median','mean','max']).round(2).to_string())

    print("\n=> If medians differ significantly: welltype is a valid metal proxy (Block 4b is meaningful).")
    print("=> If not: Block 6 (Ba-concentration strata) is the correct axis for the hypothesis.")

Metal measurements: 117 rows across 66 samples
metal      ba  fetot  mn  ni  pb
n_samples  65     16  20   9   7

Samples with welltype + metal data: 66
Welltype distribution in this subset:
welltype
Shale Gas           49
Coal Bed Methane    17
Name: count, dtype: int64


BA by welltype (n=65):
                  count   median     mean      max
welltype                                          
Coal Bed Methane     17     4.80    22.59   159.88
Shale Gas            48  2879.15  2850.54  7094.50

FETOT by welltype (n=16):
                  count  median   mean   max
welltype                                    
Coal Bed Methane      8    0.78  10.46  56.0
Shale Gas             8   20.80  21.63  43.6

MN by welltype (n=20):
                  count  median  mean    max
welltype                                    
Coal Bed Methane     13    0.31  0.40   1.45
Shale Gas             7    1.30  4.43  23.30

NI by welltype (n=9):
                  count  median  mean   max
welltype             

In [16]:
import numpy as np
import pandas as pd
# Merge genus counts with sample metadata and compute Shannon entropy
if SPARK:
    # Add welltype to genus counts
    genus_with_env = genus_pd.merge(sample_meta, on='sample_id', how='inner')
    print(f"Genus counts with welltype: {len(genus_with_env):,} rows")
    print(f"Genera with welltype data: {genus_with_env['genus'].nunique():,}")
    
    # Pivot to genus × welltype matrix
    genus_env_matrix = genus_with_env.pivot_table(
        index='genus', columns='welltype', values='total_count', 
        aggfunc='sum', fill_value=0
    )
    
    # Filter to genera present in ≥5 samples total
    genus_env_matrix = genus_env_matrix[genus_env_matrix.sum(axis=1) >= 100]  # min 100 reads
    print(f"Genera after filtering: {len(genus_env_matrix)}")
    
    # Compute Shannon entropy
    props = genus_env_matrix.div(genus_env_matrix.sum(axis=1), axis=0)
    n_envs = len(props.columns)
    H = -(props * np.log(props + 1e-300)).sum(axis=1)
    H_std = H / np.log(n_envs) if n_envs > 1 else H
    
    niche = pd.DataFrame({
        'genus': H_std.index,
        'biome_H_std': H_std.values,
        'n_envs_present': (props > 0).sum(axis=1).values
    })
    
    print(f"\nNiche breadth computed for {len(niche)} genera")
    print(f"H_std range: {niche['biome_H_std'].min():.3f} – {niche['biome_H_std'].max():.3f}")
    print(f"Welltypes: {list(props.columns)}")
    
    # Save for PGLS
    niche['genus_lower'] = niche['genus'].str.lower()
    niche.to_csv('../data/netl_genus_welltype_niche.csv', index=False)
    print(f"\nSaved: netl_genus_welltype_niche.csv")

Genus counts with welltype: 32,928 rows
Genera with welltype data: 1,562
Genera after filtering: 937

Niche breadth computed for 937 genera
H_std range: -0.000 – 0.766
Welltypes: ['Coal Bed Methane', 'Shale Gas', 'Shale_Gas', 'Tight Oil']

Saved: netl_genus_welltype_niche.csv


## Block 4b — Corrected niche metric (re-run required)

**Three methodological fixes vs. Block 4:**
1. `"Shale_Gas"` and `"Shale Gas"` merged (data quality — two labels for the same category created a phantom 4th environment)
2. `"Undefined"` welltype excluded
3. Prevalence-based Levins' B replaces read-count–proportional Shannon H — uses binary detection rate per welltype per genus (same formula as NB14 soil analysis)

**Note**: Only 3 well-type categories (Coal Bed Methane, Shale Gas, Tight Oil). Niche metric resolution is low; most genera will land in 2–3 categories with B_std compressed toward 1. Run Block 4a diagnostic first to assess whether well type is a valid metal proxy before interpreting these PGLS results.

In [17]:
# Block 4b — Corrected niche metric: prevalence-based Levins' B over well types
# Three fixes vs. Block 4:
#   1. Merge "Shale_Gas" typo into "Shale Gas" (data quality)
#   2. Exclude "Undefined" welltype
#   3. Prevalence = detection rate per welltype (not summed read counts)
if SPARK:
    # Step 1: normalize welltype labels
    sample_meta_fixed = sample_meta.copy()
    sample_meta_fixed['welltype'] = (sample_meta_fixed['welltype']
                                     .str.strip().replace('Shale_Gas', 'Shale Gas'))
    VALID_WELLTYPES = ['Coal Bed Methane', 'Shale Gas', 'Tight Oil']
    sample_meta_fixed = sample_meta_fixed[sample_meta_fixed['welltype'].isin(VALID_WELLTYPES)]
    welltype_totals = sample_meta_fixed['welltype'].value_counts().to_dict()
    print("Welltype counts after normalization:", welltype_totals)

    # Step 2: binary detection per sample
    gc_fixed = genus_pd.merge(sample_meta_fixed, on='sample_id', how='inner')
    gc_fixed['detected'] = (gc_fixed['total_count'] > 0).astype(int)

    # Step 3: prevalence per genus × welltype
    det = (gc_fixed.groupby(['genus','welltype'])['detected'].sum()
           .reset_index(name='n_detections'))
    det['n_total'] = det['welltype'].map(welltype_totals)
    det['prevalence'] = det['n_detections'] / det['n_total']

    # Step 4: pivot
    prev_mat = det.pivot_table(
        index='genus', columns='welltype', values='prevalence', fill_value=0.0
    ).reindex(columns=VALID_WELLTYPES, fill_value=0.0)

    # Step 5: filter — detected in ≥2 welltypes with ≥5 total binary detections
    n_wt_detected = (prev_mat > 0).sum(axis=1)
    total_det = gc_fixed.groupby('genus')['detected'].sum().reindex(prev_mat.index, fill_value=0)
    keep = (n_wt_detected >= 2) & (total_det >= 5)
    prev_filt = prev_mat[keep].copy()
    print(f"Genera in ≥2 welltypes with ≥5 detections: {len(prev_filt)}")

    # Step 6: Levins' B_std (same formula as NB14 soil analysis)
    row_sums = prev_filt.sum(axis=1)
    q = prev_filt.div(row_sums, axis=0)
    B = 1.0 / (q**2).sum(axis=1)
    n_envs_per_genus = (prev_filt > 0).sum(axis=1)
    B_std = (B - 1) / (n_envs_per_genus - 1).clip(lower=1)

    niche_lev = pd.DataFrame({
        'genus': B_std.index,
        'biome_H_std': B_std.values,
        'n_welltypes': n_envs_per_genus[B_std.index].values
    })
    niche_lev['genus_lower'] = niche_lev['genus'].str.lower()

    print(f"B_std range: {niche_lev['biome_H_std'].min():.3f} – {niche_lev['biome_H_std'].max():.3f}")
    print("Welltypes per genus:", niche_lev['n_welltypes'].value_counts().sort_index().to_dict())
    niche_lev.to_csv('../data/netl_levins_b_welltype.csv', index=False)
    print("Saved: netl_levins_b_welltype.csv")

Welltype counts after normalization: {'Shale Gas': 399, 'Coal Bed Methane': 332, 'Tight Oil': 155}
Genera in ≥2 welltypes with ≥5 detections: 652
B_std range: 0.011 – 0.998
Welltypes per genus: {2: 506, 3: 146}
Saved: netl_levins_b_welltype.csv


In [18]:
# Corrected PGLS — Levins' B (welltype) vs metal per-Mb + ABR negative control
# Run this after the corrected Levins' B cell above produces niche_lev
if SPARK and 'niche_lev' in dir():
    import subprocess
    from scipy.stats import zscore

    trait_table = pd.read_csv('../data/genus_trait_table.csv')
    genome_size = pd.read_csv('../data/genus_genome_size_gtdb.csv')
    abr = pd.read_csv('../data/antibiotic_resistance_genus.csv')

    m = niche_lev[['genus_lower','biome_H_std']].copy()
    m = m.merge(trait_table[['genus_lower','mean_n_metal_types','mean_n_defense_clusters',
                               'mean_n_homeostasis_clusters']],
                on='genus_lower', how='inner')
    m = m.merge(genome_size[['genus_lower','mean_genome_size_bp']], on='genus_lower', how='inner')
    m['genome_mb'] = m['mean_genome_size_bp'] / 1e6
    for col, out in [('mean_n_metal_types','ko_per_mb_total'),
                     ('mean_n_defense_clusters','ko_per_mb_tier1'),
                     ('mean_n_homeostasis_clusters','ko_per_mb_tier2')]:
        m[out] = m[col] / m['genome_mb']
    m = m.merge(abr[['genus_lower','n_antibiotic_clusters']], on='genus_lower', how='left')
    m['n_antibiotic_clusters'] = m['n_antibiotic_clusters'].fillna(0)
    m['abr_per_mb'] = m['n_antibiotic_clusters'] / m['genome_mb']
    for col in ['ko_per_mb_total','ko_per_mb_tier1','ko_per_mb_tier2','abr_per_mb']:
        m[f'{col}_z'] = zscore(m[col].values, nan_policy='omit')

    pgls_c = m[['genus_lower','biome_H_std',
                'ko_per_mb_total_z','ko_per_mb_tier1_z','ko_per_mb_tier2_z','abr_per_mb_z']].dropna()
    pgls_c.to_csv('../data/netl_pgls_input_corrected.csv', index=False)
    print(f"Corrected PGLS input: {len(pgls_c)} genera")

    R_BIN = '/home/hmacgregor/r_env/bin/Rscript'
    res = subprocess.run([
        R_BIN, '../scripts/pgls_mgnify_validation.R',
        '../data/netl_pgls_input_corrected.csv',
        '../data/gtdb_bac_genus_pruned.tree',
        '../data/netl_pgls_welltype_corrected.csv'
    ], capture_output=True, text=True)
    print(res.stdout[-3000:])
    if res.returncode != 0:
        print("STDERR:", res.stderr[-500:])

Corrected PGLS input: 267 genera



=== MGnify MAG validation PGLS ===
Input:  ../data/netl_pgls_input_corrected.csv
Tree:   ../data/gtdb_bac_genus_pruned.tree
Output: ../data/netl_pgls_welltype_corrected.csv

Loaded 267 genera from input CSV
Tree has 2283 tips
After tree pruning: 258 genera

Predictors to test (4): ko_per_mb_total_z, ko_per_mb_tier1_z, ko_per_mb_tier2_z, abr_per_mb_z

[PGLS] biome_H_std ~ ko_per_mb_total_z  (n=258)
  lambda=0.2970  beta=0.0227  SE=0.0205  t=1.108  p=0.2689  deltaAIC=0.77

[PGLS] biome_H_std ~ ko_per_mb_tier1_z  (n=258)
  lambda=0.2829  beta=0.0479  SE=0.0226  t=2.124  p=0.03467  deltaAIC=-2.50

[PGLS] biome_H_std ~ ko_per_mb_tier2_z  (n=258)
  lambda=0.2970  beta=0.0310  SE=0.0201  t=1.547  p=0.1231  deltaAIC=-0.40

[PGLS] biome_H_std ~ abr_per_mb_z  (n=258)
  lambda=0.2503  beta=0.0394  SE=0.0178  t=2.221  p=0.02719  deltaAIC=-2.80

Saved: ../data/netl_pgls_welltype_corrected.csv  (4 models)

Summary:
               predictor n_taxa    lambda       beta    p_value  delta_AIC
Value  ko

In [19]:
# Block 5 — Merge with 94-KO predictors and run PGLS
if SPARK:
    # Load 94-KO predictors
    trait_table = pd.read_csv('../data/genus_trait_table.csv')
    genome_size = pd.read_csv('../data/genus_genome_size_gtdb.csv')
    genome_size['genus_lower'] = genome_size['genus_lower'].str.lower()
    
    # Merge niche with predictors
    merged = niche.merge(
        trait_table[['genus_lower', 'mean_n_metal_types', 'mean_n_defense_clusters', 
                      'mean_n_homeostasis_clusters']],
        on='genus_lower', how='inner'
    )
    merged = merged.merge(genome_size[['genus_lower', 'mean_genome_size_bp']], 
                          on='genus_lower', how='inner')
    
    # Compute per-Mb metrics
    merged['genome_size_mb'] = merged['mean_genome_size_bp'] / 1e6
    for col in ['mean_n_metal_types', 'mean_n_defense_clusters', 'mean_n_homeostasis_clusters']:
        merged[f'{col}_per_mb'] = merged[col] / merged['genome_size_mb']
    
    # Z-score
    from sklearn.preprocessing import StandardScaler
    for col in ['mean_n_metal_types_per_mb', 'mean_n_defense_clusters_per_mb', 
                'mean_n_homeostasis_clusters_per_mb']:
        vals = merged[col].values.reshape(-1, 1)
        merged[f'{col}_z'] = StandardScaler().fit_transform(vals).flatten()
    
    # Rename to match R script expectations
    merged = merged.rename(columns={
        'mean_n_metal_types_per_mb_z': 'ko_per_mb_total_z',
        'mean_n_defense_clusters_per_mb_z': 'ko_per_mb_tier1_z',
        'mean_n_homeostasis_clusters_per_mb_z': 'ko_per_mb_tier2_z',
    })
    
    # Save PGLS input
    pgls_input = merged[['genus_lower', 'biome_H_std', 'ko_per_mb_total_z', 
                         'ko_per_mb_tier1_z', 'ko_per_mb_tier2_z']].dropna()
    
    out_path = '../data/netl_pgls_input.csv'
    pgls_input.to_csv(out_path, index=False)
    print(f"PGLS input: {len(pgls_input)} genera")
    print(f"Saved: {out_path}")

PGLS input: 378 genera
Saved: ../data/netl_pgls_input.csv


In [20]:
import subprocess
R_BIN = '/home/hmacgregor/r_env/bin/Rscript'
result = subprocess.run([
    R_BIN, '../scripts/pgls_mgnify_validation.R',
    '../data/netl_pgls_input.csv',
    '../data/gtdb_bac_genus_pruned.tree',
    '../data/netl_validation_pgls.csv'
], capture_output=True, text=True)
print(result.stdout)
if result.returncode != 0:
    print("STDERR:", result.stderr[-500:])


=== MGnify MAG validation PGLS ===
Input:  ../data/netl_pgls_input.csv
Tree:   ../data/gtdb_bac_genus_pruned.tree
Output: ../data/netl_validation_pgls.csv

Loaded 378 genera from input CSV
Tree has 2283 tips
After tree pruning: 363 genera

Predictors to test (3): ko_per_mb_total_z, ko_per_mb_tier1_z, ko_per_mb_tier2_z

[PGLS] biome_H_std ~ ko_per_mb_total_z  (n=363)
  lambda=0.0189  beta=-0.0068  SE=0.0107  t=-0.633  p=0.5274  deltaAIC=1.62

[PGLS] biome_H_std ~ ko_per_mb_tier1_z  (n=363)
  lambda=-0.0936  beta=-0.0057  SE=0.0052  t=-1.100  p=0.272  deltaAIC=-5.29

[PGLS] biome_H_std ~ ko_per_mb_tier2_z  (n=363)
  lambda=-0.1060  beta=-0.0091  SE=0.0089  t=-1.016  p=0.3103  deltaAIC=NA

Saved: ../data/netl_validation_pgls.csv  (3 models)

Summary:
               predictor n_taxa      lambda         beta   p_value delta_AIC
Value  ko_per_mb_total_z    363  0.01889803 -0.006763140 0.5273924  1.620411
Value1 ko_per_mb_tier1_z    363 -0.09358873 -0.005734962 0.2719878 -5.290381
Value2 ko_

## Block 6 — Metal-concentration niche (hypothesis-aligned axis)

Well type (shale gas / coal bed methane / tight oil) classifies hydrocarbon source formation, not metal level. The correct niche axis for testing this thesis is **metal exposure gradient** across samples.

NETL sample metadata includes dissolved metal concentrations (Ba, Fe, Mn, Ni, Co, Pb) for ~65–81 samples (~14% coverage). This block uses **barium (Ba)** — the best-covered metal and a key produced-water geochemical marker — to define low/mid/high exposure strata and compute Levins' B across those strata.

**Expected limitation**: sparse metal data means n_genera will be small (~20–50). If too few for reliable PGLS, the result is reported descriptively only.

In [21]:
# Block 6a — Metal-concentration niche: Levins' B over barium strata
# Barium (Ba) has the broadest sample coverage (65 samples) and is a
# key produced-water metal linked to formation geochemistry.
# This is the hypothesis-aligned niche axis (metal exposure gradient),
# but sparse coverage means n_genera will be small.
if SPARK:
    metals_long_b6 = spark.sql("""
        SELECT sample_id, attribute as metal, try_cast(value AS DOUBLE) as conc
        FROM netl_pw_dna.study_sample_metadata
        WHERE attribute IN ('ba','fetot','mn','ni')
          AND value IS NOT NULL
          AND try_cast(value AS DOUBLE) IS NOT NULL
          AND try_cast(value AS DOUBLE) > 0
    """).toPandas()

    metals_wide_b6 = metals_long_b6.pivot_table(
        index='sample_id', columns='metal', values='conc', aggfunc='mean'
    ).reset_index()

    if 'ba' in metals_wide_b6.columns:
        ba_samples = metals_wide_b6[['sample_id','ba']].dropna()
        ba_samples = ba_samples[ba_samples['ba'] > 0]
        ba_samples['ba_bin'] = pd.qcut(ba_samples['ba'], q=3,
                                        labels=['low_Ba','mid_Ba','high_Ba'])
        print(f"Samples with Ba: {len(ba_samples)}")
        print(ba_samples.groupby('ba_bin')['ba'].agg(['count','min','median','max']).round(1))

        gc_ba = genus_pd.merge(ba_samples[['sample_id','ba_bin']], on='sample_id', how='inner')
        gc_ba['detected'] = (gc_ba['total_count'] > 0).astype(int)
        bin_totals = gc_ba['ba_bin'].value_counts().to_dict()

        # Fix: convert categorical to string before computing bin_totals
        bin_totals = gc_ba['ba_bin'].astype(str).value_counts().to_dict()
        
        det_ba = (gc_ba.groupby(['genus','ba_bin'])['detected'].sum()
                  .reset_index(name='n_det'))
        det_ba['n_total'] = det_ba['ba_bin'].astype(str).map(bin_totals)
        det_ba['prevalence'] = det_ba['n_det'] / det_ba['n_total']

        BA_BINS = ['low_Ba','mid_Ba','high_Ba']
        prev_ba = det_ba.pivot_table(
            index='genus', columns='ba_bin', values='prevalence', fill_value=0.0
        ).reindex(columns=BA_BINS, fill_value=0.0)

        # Filter: ≥2 bins detected, ≥3 total detections (sparse data → relaxed threshold)
        n_bins = (prev_ba > 0).sum(axis=1)
        total_d_ba = gc_ba.groupby('genus')['detected'].sum().reindex(prev_ba.index, fill_value=0)
        keep_ba = (n_bins >= 2) & (total_d_ba >= 3)
        prev_ba_filt = prev_ba[keep_ba].copy()
        print(f"\nGenera in ≥2 Ba strata with ≥3 detections: {len(prev_ba_filt)}")

        # Levins' B_std
        rs_ba = prev_ba_filt.sum(axis=1)
        q_ba = prev_ba_filt.div(rs_ba, axis=0)
        B_ba = 1.0 / (q_ba**2).sum(axis=1)
        n_ba = (prev_ba_filt > 0).sum(axis=1)
        B_ba_std = (B_ba - 1) / (n_ba - 1).clip(lower=1)

        niche_ba = pd.DataFrame({'genus': B_ba_std.index, 'biome_H_std': B_ba_std.values})
        niche_ba['genus_lower'] = niche_ba['genus'].str.lower()
        print(f"B_std range: {niche_ba['biome_H_std'].min():.3f} – {niche_ba['biome_H_std'].max():.3f}")
        niche_ba.to_csv('../data/netl_levins_b_ba_strata.csv', index=False)
        print("Saved: netl_levins_b_ba_strata.csv")
    else:
        print("ERROR: 'ba' column not in metal data")

Samples with Ba: 65
         count     min  median     max
ba_bin                                
low_Ba      22     1.4     5.1    28.2
mid_Ba      21    65.1  1983.1  2940.1
high_Ba     22  3082.3  3952.6  7094.5

Genera in ≥2 Ba strata with ≥3 detections: 180
B_std range: 0.296 – 1.000
Saved: netl_levins_b_ba_strata.csv


In [22]:
# Block 6b — PGLS: Levins' B over Ba strata vs metal gene density
# Small n expected (~20-50 genera) due to sparse metal data — interpret cautiously.
if SPARK and 'niche_ba' in dir() and len(niche_ba) >= 10:
    from scipy.stats import zscore
    import subprocess

    trait_table = pd.read_csv('../data/genus_trait_table.csv')
    genome_size = pd.read_csv('../data/genus_genome_size_gtdb.csv')
    abr = pd.read_csv('../data/antibiotic_resistance_genus.csv')

    m_ba = niche_ba[['genus_lower','biome_H_std']].copy()
    m_ba = m_ba.merge(trait_table[['genus_lower','mean_n_metal_types','mean_n_defense_clusters',
                                    'mean_n_homeostasis_clusters']],
                      on='genus_lower', how='inner')
    m_ba = m_ba.merge(genome_size[['genus_lower','mean_genome_size_bp']], on='genus_lower', how='inner')
    m_ba['genome_mb'] = m_ba['mean_genome_size_bp'] / 1e6
    for col, out in [('mean_n_metal_types','ko_per_mb_total'),
                     ('mean_n_defense_clusters','ko_per_mb_tier1'),
                     ('mean_n_homeostasis_clusters','ko_per_mb_tier2')]:
        m_ba[out] = m_ba[col] / m_ba['genome_mb']
    m_ba = m_ba.merge(abr[['genus_lower','n_antibiotic_clusters']], on='genus_lower', how='left')
    m_ba['n_antibiotic_clusters'] = m_ba['n_antibiotic_clusters'].fillna(0)
    m_ba['abr_per_mb'] = m_ba['n_antibiotic_clusters'] / m_ba['genome_mb']
    for col in ['ko_per_mb_total','ko_per_mb_tier1','ko_per_mb_tier2','abr_per_mb']:
        m_ba[f'{col}_z'] = zscore(m_ba[col].values, nan_policy='omit')

    pgls_ba = m_ba[['genus_lower','biome_H_std',
                    'ko_per_mb_total_z','ko_per_mb_tier1_z','ko_per_mb_tier2_z','abr_per_mb_z']].dropna()
    pgls_ba.to_csv('../data/netl_pgls_input_ba_strata.csv', index=False)
    print(f"Ba-strata PGLS input: {len(pgls_ba)} genera")

    if len(pgls_ba) >= 10:
        R_BIN = '/home/hmacgregor/r_env/bin/Rscript'
        res = subprocess.run([
            R_BIN, '../scripts/pgls_mgnify_validation.R',
            '../data/netl_pgls_input_ba_strata.csv',
            '../data/gtdb_bac_genus_pruned.tree',
            '../data/netl_pgls_ba_strata.csv'
        ], capture_output=True, text=True)
        print(res.stdout[-3000:])
        if res.returncode != 0:
            print("STDERR:", res.stderr[-500:])
    else:
        print(f"Only {len(pgls_ba)} genera — too few for reliable PGLS.")
        print("Interpret descriptively: do high-metal genera show more metal genes than low-metal?")
        print(m_ba[['genus_lower','ko_per_mb_total','biome_H_std']].sort_values('biome_H_std').to_string())
else:
    print("Ba strata niche not computed or <10 genera — skipping PGLS.")

Ba-strata PGLS input: 77 genera



=== MGnify MAG validation PGLS ===
Input:  ../data/netl_pgls_input_ba_strata.csv
Tree:   ../data/gtdb_bac_genus_pruned.tree
Output: ../data/netl_pgls_ba_strata.csv

Loaded 77 genera from input CSV
Tree has 2283 tips
After tree pruning: 73 genera

Predictors to test (4): ko_per_mb_total_z, ko_per_mb_tier1_z, ko_per_mb_tier2_z, abr_per_mb_z

[PGLS] biome_H_std ~ ko_per_mb_total_z  (n=73)
  lambda=-0.7506  beta=0.0451  SE=0.0150  t=3.015  p=0.00356  deltaAIC=-3.18

[PGLS] biome_H_std ~ ko_per_mb_tier1_z  (n=73)

STDERR: Error in data.frame(response = response_col, predictor = predictor_col,  : 
  arguments imply differing number of rows: 1, 0
Calls: Filter ... unlist -> lapply -> lapply -> FUN -> run_pgls -> data.frame
Execution halted

